# Categorização Automática de Demandas de Ouvidoria
### Trabalho final — Deep Learning e PLN (IDP) · Modalidade 2 (NLP no Setor Público)

**Integrantes:** _(preencher)_
**Data:** _(preencher)_

---

**Problema.** Direcionar automaticamente manifestações de cidadãos para a área responsável
(Saneamento, Iluminação, Trânsito, Saúde, etc.) a partir do texto livre da reclamação.
Tarefa: **classificação de texto multiclasse** em português do Brasil.

**Hipótese central.** Um modelo de Deep Learning (Transformer com fine-tuning) supera o
baseline clássico (TF-IDF + modelo linear) na métrica **F1-macro**.

> Notebook **orquestrador**: a lógica pesada vive em `src/`; aqui apenas importamos, chamamos e narramos.


## 0. Setup e configuração

Imports gerais, seed e caminhos. Ver `requirements.txt` para o ambiente.

In [ ]:
import sys, os, json, random
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Permite importar de src/
sys.path.append(str(Path.cwd().parent))

DATA_RAW = Path("../data/raw")
DATA_INTERIM = Path("../data/interim")
DATA_PROCESSED = Path("../data/processed")
RESULTADOS = Path("../resultados")
for p in (DATA_RAW, DATA_INTERIM, DATA_PROCESSED, RESULTADOS):
    p.mkdir(parents=True, exist_ok=True)

print("Ambiente configurado. Seed =", RANDOM_STATE)

## 1. Coleta de dados

> ⚠️ **Regra crítica:** dados devem ser **coletados pela equipe** (scraping/API), não baixados prontos.
> Antes de rodar scraping ao vivo: checar `robots.txt`, aplicar *delay*, identificar *user-agent*,
> anonimizar dados pessoais e registrar o log da coleta.

Descreva aqui: **fonte escolhida** (Fala.BR / 156 municipal / Relato do Consumidor), período,
filtros e volume coletado.

In [ ]:
# from src.coleta.coletor import coletar

# TODO: implementar a coleta em src/coleta/coletor.py e chamá-la aqui.
# df_bruto = coletar(fonte="...", periodo=("2026-01-01", "2026-05-31"))
# df_bruto.to_parquet(DATA_RAW / "coleta.parquet", index=False)

# Carregar (após coletar):
# df_bruto = pd.read_parquet(DATA_RAW / "coleta.parquet")
# print(df_bruto.shape); df_bruto.head()

## 2. Rotulagem e concordância entre anotadores

Defina de **5 a 10 categorias** claras. Se a fonte já traz "assunto/problema", use como rótulo;
senão, rotule manualmente uma amostra com **dois anotadores** e meça o **kappa de Cohen**
(diferencial metodológico).

In [ ]:
# from sklearn.metrics import cohen_kappa_score

# CATEGORIAS = ["Saneamento", "Iluminacao", "Transito", "Saude", "Limpeza", "Outros"]

# TODO: carregar rótulos dos dois anotadores e medir concordância
# kappa = cohen_kappa_score(anotador_1, anotador_2)
# print(f"Kappa de Cohen: {kappa:.3f}")

# Salvar base rotulada:
# df_rotulado.to_parquet(DATA_INTERIM / "rotulado.parquet", index=False)

## 3. Pré-processamento

> Limpeza **agressiva** para o baseline TF-IDF (lowercase, stopwords, lematização).
> Limpeza **leve** para o Transformer (preservar contexto — o BERT usa a sentença inteira).

Faça também o **split estratificado** treino/validação/teste (estratificar pela categoria,
pois há desbalanceamento).

In [ ]:
# from src.preprocessamento.limpeza import limpar_para_tfidf, limpar_para_bert
# from sklearn.model_selection import train_test_split

# df = pd.read_parquet(DATA_INTERIM / "rotulado.parquet")

# TODO: aplicar limpezas e split estratificado
# X_train, X_temp, y_train, y_temp = train_test_split(
#     df["texto"], df["categoria"], test_size=0.3,
#     stratify=df["categoria"], random_state=RANDOM_STATE)
# X_val, X_test, y_val, y_test = train_test_split(
#     X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE)

## 4. Baseline clássico — TF-IDF + modelo linear

Ponto de comparação **obrigatório**. Rápido e interpretável. É contra este número que o
Deep Learning precisa provar valor.

In [ ]:
# from src.modelos.baseline import treinar_baseline
# from sklearn.metrics import classification_report, f1_score

# TODO: TF-IDF + LogisticRegression (ou SVM)
# modelo_base, pred_base = treinar_baseline(X_train, y_train, X_test)
# f1_base = f1_score(y_test, pred_base, average="macro")
# print(f"F1-macro baseline: {f1_base:.3f}")
# print(classification_report(y_test, pred_base))

## 5. Modelo de Deep Learning — fine-tuning de Transformer

Modelo base: **BERTimbau** (`neuralmind/bert-base-portuguese-cased`).
Opcional comparar com BERTugues / Albertina PT-BR. Se a GPU for limitada, usar **LoRA/QLoRA** (`peft`).

In [ ]:
# from src.modelos.transformer import treinar_transformer

# MODELO = "neuralmind/bert-base-portuguese-cased"

# TODO: tokenização, Trainer/TrainingArguments, fine-tuning
# modelo_dl, pred_dl = treinar_transformer(
#     modelo=MODELO, X_train=X_train, y_train=y_train,
#     X_val=X_val, y_val=y_val, X_test=X_test, seed=RANDOM_STATE)
# f1_dl = f1_score(y_test, pred_dl, average="macro")
# print(f"F1-macro Transformer: {f1_dl:.3f}")

## 6. Avaliação comparativa

> **Priorize F1-macro** (dados desbalanceados), não acurácia. Inclua precisão/revocação por
> classe e a **matriz de confusão** de cada modelo.

In [ ]:
# from src.avaliacao.metricas import comparar_modelos, plotar_matriz_confusao

# TODO: tabela comparativa baseline x Transformer + matrizes de confusão
# resultados = {"baseline": f1_base, "transformer": f1_dl, "ganho": f1_dl - f1_base}
# json.dump(resultados, open(RESULTADOS / "metricas.json", "w"), indent=2, ensure_ascii=False)
# print(resultados)

## 7. Diferencial — categoria prevista × geografia

O "algo a mais" valorizado no plano: cruzar a categoria prevista pelo modelo com o **bairro/CEP**
da reclamação e gerar um **mapa de calor** de onde o poder público deveria alocar recursos por tipo
de demanda.

In [ ]:
# import folium, geopandas as gpd

# TODO: agregar previsões por região e plotar mapa coroplético / de calor
# salvar em RESULTADOS / "figuras/mapa_demandas.html" 

## 8. Conclusão e próximos passos

- O Deep Learning superou o baseline? Em quanto (F1-macro)?
- Traduza o ganho em **impacto de serviço público** (triagem mais rápida, menos retrabalho,
  alocação de recursos por região).
- Limitações (tamanho/representatividade da amostra, vieses, categorias ambíguas).
- Próximos passos (mais dados, modelos maiores, humano no loop).

---
**Referências** — ver `docs/referencias.md` (mínimo 5 de domínio + 5 de técnica, ABNT).